[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/exercices/seance2_exercices.ipynb)

# Séance 2.2 — Nettoyer des données réelles

**Exercices** · durée : 2h — cinq defauts, chacun suivi de deux exercices

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- repérer les six défauts classiques d'un fichier réel
- convertir du texte en nombres et en dates
- traiter les valeurs manquantes en connaissance de cause
- supprimer les doublons et écarter les valeurs aberrantes
- classer des lignes selon une règle, sur la colonne entière
- construire un pipeline de nettoyage qu'on peut rejouer

## Pour aller plus loin

Les exercices de la séance sont dans votre **notebook de cours** : c'est là qu'on
travaille ensemble. Cette feuille-ci est **facultative**.

Elle reprend les mêmes techniques sur d'autres questions — à faire quand vous avez fini
avant les autres, ou tranquillement après la séance. Certains exercices ont des `____` à
remplir, d'autres une cellule vide où vous écrivez tout ; ceux qui se terminent par une
cellule de **vérification** vous disent immédiatement si votre réponse est bonne.

> 💡 Pas de vérification partout. Affichez systématiquement votre résultat et demandez-vous
> s'il est **plausible** : c'est le seul contrôle dont vous disposerez en entreprise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut

# Le pipeline de la seance, rejoue d'un coup : c'est le point de depart
# des questions ci-dessous. Meme ordre qu'en seance.
net = sale.drop_duplicates().dropna(subset=["client_id"]).copy()
net["prix"] = pd.to_numeric(net["prix"].str.replace(" EUR", "")
                            .str.replace(",", "."))
net["date"] = pd.to_datetime(net["date"], format="mixed", dayfirst=True)
net["categorie"] = net["categorie"].str.strip().str.lower()

print(len(sale), "->", len(net))

### Exercice 1 — Extraire le mois

> **Votre mission :**
> - Créer la colonne `mois` de `net` à partir de `date`.
> - Mettre le numéro du mois qui compte le plus de lignes dans `mois_top`.

In [ ]:
net["mois"] = net["date"].____.month
mois_top = net["mois"].value_counts().____()

print(mois_top)

In [ ]:
verifier("1 - mois le plus charge", mois_top == 10,
         ".dt.month sur une colonne de dates, puis value_counts().idxmax()")

### Exercice 2 — Classer selon plusieurs conditions — `np.select`

> **Votre mission :**
> - `np.where` ne tranche qu'entre deux cas. `np.select` en accepte autant qu'on veut.
> - Il attend trois choses : une **liste de conditions** (dans l'ordre où on veut qu'elles soient essayées), une **liste d'étiquettes** de même longueur, et `default=...` pour les lignes qu'aucune condition n'a retenues.
> - Créer `net['ca']`, puis une colonne `taille` : « tres grosse » au-dessus de 200 €, « grosse » au-dessus de 50 €, « petite » sinon. Compter les « tres grosse » → `nb_tres_grosses`.
> - ⚠️ **C'est la première condition vraie qui l'emporte** : de la plus restrictive à la plus large. Dans l'autre ordre, « tres grosse » resterait vide sans qu'aucune erreur ne le signale.

In [ ]:
net["ca"] = net["qte"] * net["prix"]

conditions = [net["ca"] > ____, net["ca"] > ____]   ## ordre = priorite
etiquettes = ["tres grosse", "grosse"]

net["taille"] = np.select(conditions, etiquettes, default="petite")
nb_tres_grosses = (net["taille"] == "tres grosse").sum()

print(net["taille"].value_counts())

In [ ]:
verifier("2 - lignes tres grosses", nb_tres_grosses == 59,
         "conditions dans l'ordre 200 puis 50 : la premiere vraie l'emporte")

### Exercice 3 — Découper en tranches — `pd.cut`

> **Votre mission :**
> - Ranger les prix en quatre gammes : `entree` (0 à 1 €), `eco` (1 à 5 €), `milieu` (5 à 20 €), `premium` (au-delà).
> - Combien de références tombent en `entree` → `nb_entree` ?
> - *Nouveau :* `pd.cut(col, bins=[...], labels=[...])` — **cinq bornes** délimitent **quatre** tranches.

In [ ]:
net["gamme"] = pd.cut(net["prix"],
                      bins=[0, 1, 5, 20, 10000],
                      labels=["entree", "____", "milieu", "premium"])
nb_entree = (net["gamme"] == "entree").sum()

print(net["gamme"].value_counts())

In [ ]:
verifier("3 - references d'entree de gamme", nb_entree == 1093,
         "quatre etiquettes pour cinq bornes, dans l'ordre croissant")

### Question 4 — Le prix d'un `dropna()` négligent

> **Votre mission :**
> - Combien de lignes resteraient après un `dropna()` **sans argument** sur `sale` ?
> - Et après `dropna(subset=['client_id'])` ?
> - Ici les deux donnent le même résultat. Dans quel cas seraient-ils très différents ?

### Question 5 — Les aberrations, sans seuil arbitraire

> **Votre mission :**
> - En séance, on a écarté `qte >= 10000` — un seuil choisi à la main.
> - Refaire le repérage avec la **règle de l'écart interquartile** : est aberrant ce qui dépasse `q3 + 1.5 * (q3 - q1)`.
> - Combien de lignes dépasse-t-elle ? Faut-il toutes les supprimer ?

### Question 6 — Le compte rendu qualité

> **Votre mission :**
> - Vous rendez le fichier nettoyé. Produire les **quatre chiffres** de la note d'accompagnement :
> - lignes au départ · lignes conservées · taux de perte en % · **part du chiffre d'affaires réel** que représentent les ventes écartées faute de client identifié.
> - Ce dernier chiffre est celui qu'on oublie. Comparez-le au taux de perte en lignes : que vous dit l'écart entre les deux ?
> - ⚠️ Calculez ce CA sur les ventes **plausibles** uniquement. Les quantités à 99 999 produiraient sinon un total fictif qui écrase tout le reste.